In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import json
import joblib
from itertools import combinations
from scipy.stats import randint
from functools import partial

from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, recall_score, fbeta_score, make_scorer
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, TunedThresholdClassifierCV, TimeSeriesSplit
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

import sys
from pathlib import Path

In [8]:
pd.set_option('display.max_columns', None)

ROOT = Path.cwd()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
     ROOT = ROOT.parent
src_path = str(ROOT/"src")
if src_path not in sys.path:
    sys.path.insert(0,src_path)

from config import START_YEAR, END_YEAR, HOLDOUT_START
import utils
from final_model import ClassifiedRegressor

In [ ]:
df = utils.player_stat_dataframe(START_YEAR,END_YEAR)

train = df[df.Year < HOLDOUT_START - 3]
val =  df[df.Year == HOLDOUT_START - 3]
test = df[df.Year >= HOLDOUT_START]

req = ['Year', 'GP%', 'MP']
f = ['PTS','W%','DRB','AST','BLK','STL','FT','3P','eFG%']
features = list(set(req + f))

target = 'MVPVS'

In [ ]:
X_tr, y_tr = train[features], train[target]
X_v, y_v = val[features], val[target]

#cv = [list(utils.forward_chaining_splits(X_tr.Year, min_train_years=10))[0]]   # first running it only for [0]
cv = list(utils.forward_chaining_splits(X_tr.Year, min_train_years=10))   # all folds, not just [0]

base = ClassifiedRegressor(
    classifier=make_pipeline(
        PolynomialFeatures(degree=2), 
        StandardScaler(),
        LogisticRegression(solver='saga', max_iter=1000, random_state=97, tol=1e-3)),
    regressor=HistGradientBoostingRegressor(random_state=29),
)

# Stage 1: grid over the classifier (and the gate threshold), regressor held fixed
grid = GridSearchCV(
    base,
    param_grid={
        'classifier__logisticregression__C': np.logspace(-1.5, 1.5, 5),
        'classifier__logisticregression__l1_ratio': [0, 0.5, 1],
        'threshold': [0.3, 0.5],
    },
    cv=cv, scoring='neg_mean_squared_error', n_jobs=-1,
)
grid.fit(X_tr, y_tr)

print(f'Best parameters:')
print(grid.best_params_)


In [19]:
# Stage 2: random search over the regressor, starting from the best classifier
rand = RandomizedSearchCV(
    grid.best_estimator_,          # gets cloned, so it keeps the best classifier/threshold
    param_distributions={
        'regressor__loss': ['squared_error', 'poisson'],
        'regressor__learning_rate': [0.01, 0.03, 0.05],
        'regressor__max_iter': [100, 200],
        'regressor__max_depth': [2, 3, 4, None],
        'regressor__max_leaf_nodes': [7, 15, 31],
        'regressor__min_samples_leaf': [1, 2, 3, 4, 5],
        'regressor__l2_regularization': [0.0, 0.5, 1.0],
    },
    n_iter=200, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1, random_state=29,
)
rand.fit(X_tr, y_tr)
print(f'Best parameters:')
print(rand.best_params_)

model = rand.best_estimator_


Best parameters:
{'regressor__min_samples_leaf': 4, 'regressor__max_leaf_nodes': 7, 'regressor__max_iter': 200, 'regressor__max_depth': 3, 'regressor__loss': 'squared_error', 'regressor__learning_rate': 0.03, 'regressor__l2_regularization': 1.0}


In [21]:
res = val
res['Output'] = model.predict(X_v)
fin = res.sort_values('MVPVS', ascending=False)[['Player','MVPVS','Output']]
fin[(fin['Output']> 0) | (fin['MVPVS']> 0)]

,Player,MVPVS,Output
14295,Stephen Curry,1.000,0.981590
14296,Kawhi Leonard,0.484,0.071055
14297,LeBron James,0.482,0.291878
14298,Russell Westbrook,0.371,0.215383
14299,Kevin Durant,0.112,0.363594
14300,Chris Paul,0.082,0.104157
14301,Draymond Green,0.038,0.155853
14302,Damian Lillard,0.020,0.040085
14303,James Harden,0.007,0.237626
14304,Kyle Lowry,0.005,0.089743


In [14]:
X_tr, y_tr = train[features], train[target]
X_v, y_v = val[features], val[target]

cv = [list(utils.forward_chaining_splits(train.Year, min_train_years=10))[0]]

model = ClassifiedRegressor(
    classifier = make_pipeline(
            PolynomialFeatures(degree=2), 
            StandardScaler(), 
            LogisticRegression(solver='saga',max_iter=1000, random_state=97, tol=1e-3, C=np.float64(5.623413251903491), l1_ratio=0)),
    regressor = HistGradientBoostingRegressor(random_state=29)
)

model.fit(X_tr,y_tr)


,classifier,Pipeline(step... tol=0.001))])
,regressor,HistGradientB...ndom_state=29)
,threshold,0.5
,check_eligibility,True
Name,Type,Value
classifier_,Pipeline,Pipeline(step... tol=0.001))])
cv_,list,"[(array([ 0, ...78, 179, 180]), array([181, 1...54, 355, 356])), (array([ 0, ...54, 355, 356]), array([357, 3...36, 537, 538])), (array([ 0, ...36, 537, 538]), array([539, 5...24, 725, 726])), (array([ 0, ...24, 725, 726]), array([727, 7...04, 905, 906])), ...]"
regressor_,HistGradientBoostingRegressor,HistGradientB...ndom_state=29)
,"degree degree: int or tuple (min_degree, max_degree), default=2If a single int is given, it specifies the maximal degree of thepolynomial features. If a tuple `(min_degree, max_degree)` is passed,then `min_degree` is the minimum and `max_degree` is the maximumpolynomial degree of the generated features. Note that `min_degree=0`and `min_degree=1` are equivalent as outputting the degree zero term isdetermined by `include_bias`.",2
,"interaction_only interaction_only: bool, default=FalseIf `True`, only interaction features are produced: features that areproducts of at most `degree` *distinct* input features, i.e. terms withpower of 2 or higher of the same input feature are excluded:- included: `x[0]`, `x[1]`, `x[0] * x[1]`, etc.- excluded: `x[0] ** 2`, `x[0] ** 2 * x[1]`, etc.",False
,"include_bias include_bias: bool, default=TrueIf `True` (default), then include a bias column, the feature in whichall polynomial powers are zero (i.e. a column of ones - acts as anintercept term in a linear model).",True


In [15]:
res = val
res['Output'] = model.predict(X_v)
fin = res.sort_values('Output', ascending=False)[['Player','MVPVS','Output']]
fin[(fin['Output']> 0) | (fin['MVPVS']> 0)]

,Player,MVPVS,Output
14295,Stephen Curry,1.000,1.013654
14297,LeBron James,0.482,0.345903
14299,Kevin Durant,0.112,0.309292
14298,Russell Westbrook,0.371,0.246364
14616,Klay Thompson,0.000,0.211702
14301,Draymond Green,0.038,0.194705
14300,Chris Paul,0.082,0.193979
14303,James Harden,0.007,0.190604
14591,DeAndre Jordan,0.000,0.168370
14304,Kyle Lowry,0.005,0.136930


In [12]:
a = np.array([1,2,3])
a>1

array([False,  True,  True])